# CT VLM — evaluation notebook

Evaluate the fine-tuned CT VLM (`1308_qwen3_5_4b_atlas_3d_ct`) on the **held-out test split** across three tasks:

| task | model output | ground truth |
|---|---|---|
| **structured report generation** | JSON report (full: all systems / mini: findings,summary,complete_reasoning,impression,recommendations,prominence_score) | reference structured report |
| **cls tokens** | space-separated `<TAG>` abnormality tokens | GT tag set |
| **cls token json** | JSON abnormalities with `presence / size / location / severity / characterization` | GT abnormality JSON |

**How to use:** the parquet paths below are **DUMMY** — run inference on the `testing/` splits, save the prediction
parquets, then point the config at them. This notebook loads them and documents the metrics to compute; the metric /
LLM-judge implementations are intentionally left as stubs (fill in once the predictions exist).

> Metric ideas are written up per task in the markdown sections. LLM-as-judge uses **Intern-S2** (served via the
> `lm_deploy/` scripts) — the judge prompt is *described*, not implemented here.

In [ ]:
import os
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## Config — eval parquet paths (DUMMY) + column conventions

In [ ]:
# ---------------------------------------------------------------------------
# One prediction parquet per task. DUMMY paths -- replace after you run inference.
# Each parquet: one row per test volume, carrying the model PREDICTION and the
# GROUND TRUTH (an explicit column, or recoverable from the `messages` column).
# ---------------------------------------------------------------------------
EVAL_DIR = "/cache/fast_data_nas71/janhavi/data/ct_eval/predictions"   # <-- change me

EVAL_PARQUETS = {
    # structured report generation (two prompt variants trained: full + mini)
    "structured_report_full": f"{EVAL_DIR}/DUMMY_full_structured_report_preds.parquet",
    "structured_report_mini": f"{EVAL_DIR}/DUMMY_mini_structured_report_preds.parquet",
    # abnormality classification as <TAG> cls-tokens
    "cls_tokens":             f"{EVAL_DIR}/DUMMY_cls_tokens_preds.parquet",
    # abnormality classification as JSON (presence / size / location / characterization)
    "cls_token_json":         f"{EVAL_DIR}/DUMMY_cls_token_json_preds.parquet",
}

# Column names in the prediction parquets (adjust to whatever inference wrote).
ID_COL   = "videos"          # per-volume id (or "series_uid")
PRED_COL = "response"        # the model output (prediction)
GT_COL   = "ground_truth"    # if absent, GT is recovered from the last assistant turn in `messages`

## Loaders

Robust to the dummy paths (skips missing files). Normalizes a `_gt` / `_pred` view per row so the metric cells don't care where GT/pred physically live.

In [ ]:
def _gt_from_messages(row):
    """Recover ground truth = the last assistant/gpt turn's value (messages may be a list or a JSON string)."""
    msgs = row.get("messages")
    if isinstance(msgs, str):
        try:
            msgs = json.loads(msgs)
        except Exception:
            return None
    if isinstance(msgs, (list, tuple, np.ndarray)) and len(msgs):
        last = dict(msgs[-1])
        return last.get("value", last.get("content"))
    return None


def load_eval(path):
    if not os.path.exists(path):
        print(f"[dummy] {os.path.basename(path)} -- set the real path after inference")
        return None
    df = pd.read_parquet(path)
    df["_gt"]   = df[GT_COL] if GT_COL in df.columns else df.apply(_gt_from_messages, axis=1)
    df["_pred"] = df[PRED_COL] if PRED_COL in df.columns else None
    print(f"{os.path.basename(path)}: {len(df):,} rows | cols={list(df.columns)}")
    return df


evals = {task: load_eval(p) for task, p in EVAL_PARQUETS.items()}
# e.g. evals["cls_tokens"] -> DataFrame with _gt / _pred once real paths are set

---
## Metric ideas — overview

| family | structured report | cls tokens | cls token json |
|---|---|---|---|
| **generation health** | JSON-parse rate, schema completeness, empty/truncated rate | parse rate, avg #tags | JSON-parse rate, schema validity |
| **classification (multi-label)** | prominence-tag set P/R/F1 | micro/macro/per-tag P/R/F1, subset-acc, Jaccard | presence P/R/F1 (per abnormality) |
| **ordinal / numeric** | prominence 1–5: MAE, Spearman, quadratic-weighted κ | — | size (mm) error / within-tolerance; severity κ |
| **text / semantic** | ROUGE-L, BLEU, METEOR, BERTScore, **report-embedding cosine** on findings/impression | — | characterization: embedding sim / LLM |
| **localization** | (via judge) | — | laterality/lung/lobe/segment exact & partial match |
| **LLM-as-judge (Intern-S2)** | finding-level TP/FP/FN → P/R/F1; Likert sub-scores | (not needed — fixed vocab) | attribute correctness, hallucination rate |

Details per task below.

## Task 1 — Structured report generation

**a) Generation health (deterministic)**
- **JSON validity / parse rate** — % predictions that parse (strip ```json fences first).
- **Schema completeness** — fraction of expected top-level keys present (full: all systems; mini: the 6 fields). Per-field population/non-empty rate.
- **Empty / truncated rate** — blank output, or JSON cut off by the token budget.

**b) Abnormality agreement via `prominence_score` (multi-label + ordinal)**
- Predicted vs GT **tag set** → precision / recall / F1 (micro + macro + per-tag). This is the most clinically meaningful deterministic signal (the prominence dict names the positive findings).
- For tags present in **both**: prominence (1–5) agreement — **MAE**, **Spearman ρ**, **quadratic-weighted Cohen's κ**.

**c) Free-text fields (findings / summary / impression / recommendations)**
- Lexical: **ROUGE-1/2/L**, **BLEU**, **METEOR** vs the reference field.
- Semantic: **BERTScore**, and — best for this domain — **cosine similarity of your CT report-embedding model** (`vlm/training/report_embedding`) between predicted and GT findings/impression. Reuses a model already tuned on these reports.
- Watch length bias: report mean/median length pred vs GT; normalize or report alongside.

**d) Section-level structural match**
- For each anatomical section (lungs, pleura, mediastinum, …): normal-vs-abnormal agreement (binary), and negation handling (did the model correctly say "no X"?). Guards against a model that scores well on text overlap by parroting boilerplate negatives.

**e) LLM-as-judge (Intern-S2)** — see the LLM-judge section. The key quantifiable output: judge-extracted **finding-level TP/FP/FN → precision/recall/F1** (recall = did it catch GT findings, precision = did it avoid hallucinating), plus Likert sub-scores for localization/size/impression.

In [ ]:
# --- Task 1 metrics: implement here (see markdown above) ---
# df = evals["structured_report_full"]  # or ["structured_report_mini"]
# steps: parse JSON -> schema completeness ; prominence-tag P/R/F1 + score agreement ;
#        ROUGE / report-embedding cosine on findings+impression ; then LLM-judge finding-level F1.
pass

## Task 2 — CLS tokens (multi-label classification)

Output is `<TAG_A> <TAG_B> ...` from a **fixed vocabulary**, so this is clean multi-label classification — no LLM judge needed.

- Parse both pred and GT to tag sets (regex `<([A-Z_]+)>`; normalize case/underscores; drop unknown tokens).
- Build the binary label matrix over the tag vocabulary, then:
  - **Micro / macro / weighted precision, recall, F1** (sklearn `classification_report` / `precision_recall_fscore_support`).
  - **Per-tag F1 + support** — surface the best/worst tags and rare-tag behavior.
  - **Subset (exact-match) accuracy** and **example-based F1** (mean per-row F1).
  - **Jaccard / IoU** and **Hamming loss** of the tag sets.
  - **Label cardinality** — avg #tags predicted vs GT (over/under-prediction bias).
- **Hierarchical view:** roll tags up to the 12 categories (nodules, vascular, bone, …) and report category-level P/R/F1 too — partial credit for right-category / wrong-subtype.
- **Calibration of thresholds** isn't applicable (discrete tokens), but log the most over- and under-predicted tags for error analysis.

In [ ]:
# --- Task 2 metrics: implement here ---
# df = evals["cls_tokens"]
# import re; TAG = re.compile(r"<([A-Z0-9_]+)>")
# pred_tags = df["_pred"].map(lambda s: set(TAG.findall(str(s))))
# gt_tags   = df["_gt"].map(lambda s: set(TAG.findall(str(s))))
# -> binarize over the tag vocab, then sklearn multilabel P/R/F1 (micro/macro/per-tag), subset-acc, Jaccard.
pass

## Task 3 — CLS token JSON (presence + attributes)

Output is JSON: `{abnormality: {presence, size, location{laterality,lung,lobe,segment,...}, severity, detailed_characteristics}}`.

**a) Health**
- JSON parse rate; schema validity (expected nesting); presence field coercible to bool.

**b) Presence (multi-label)** — same as cls-tokens but derived from `presence == true`: per-abnormality **P/R/F1**, subset-acc, Jaccard. Lets you check whether the JSON head agrees with the cls-token head on the *same* volumes (consistency between the two abnormality formats).

**c) Attribute agreement (only over abnormalities present in BOTH pred & GT)**
- **Location:** exact and partial match on `laterality / lung / lobe / segment` (e.g. fraction of location sub-fields matching; a hierarchical score that gives partial credit for right lung / wrong lobe).
- **Size:** parse the numeric mm (regex), report **MAE**, **% within ±tolerance** (e.g. ±2 mm or ±25%), and a Bland–Altman style bias; or bucket to size categories and use κ.
- **Severity:** ordinal agreement (κ) if it maps to an ordered scale, else exact-match.
- **Characterization (free text):** embedding cosine (report-embedding model) or defer to the LLM judge.

**d) LLM-as-judge** — for characterization correctness and to catch semantically-equal-but-lexically-different attributes.

In [ ]:
# --- Task 3 metrics: implement here ---
# df = evals["cls_token_json"]
# steps: parse JSON -> presence P/R/F1 ; for matched abnormalities: location exact/partial match,
#        size mm error / within-tolerance, severity kappa, characterization embedding/LLM sim.
pass

---
## LLM-as-judge with Intern-S2 (design)

Use Intern-S2 (served via `data_preparation/ct_vlm/llm_inference/lm_deploy/`) as a **reference-based** judge: it never sees images, only the **prediction** and the **ground-truth report/JSON**, and scores agreement. Reference-based (not open-ended) keeps it grounded and cheap.

### Where a judge adds value (vs deterministic metrics)
- Semantic equivalence text metrics miss ("3 mm RUL nodule" == "tiny nodule in right upper lobe").
- **Finding-level** correctness with synonymy/negation handled properly.
- Free-text `detailed_characteristics` / `impression` quality.

### Make the judge output QUANTIFIABLE (not vibes)
Prefer **counts and fixed scales** that reduce to standard metrics:

1. **Finding-level TP/FP/FN (best).** Ask the judge to align predicted findings to GT findings and return integer counts:
   `true_positives` (GT finding correctly reported), `false_negatives` (GT finding missed), `false_positives` (reported but not in GT / hallucinated). Then aggregate → **precision = TP/(TP+FP)**, **recall = TP/(TP+FN)**, **F1**, micro-averaged over the test set. This is the headline metric and directly interpretable as "clinical sensitivity/precision".
2. **Attribute-conditioned correctness.** For each matched finding, boolean flags: `location_correct`, `size_correct`, `severity_correct` → report accuracy of each.
3. **Likert sub-scores (1–5), fixed rubric:** `completeness` (recall of GT findings), `correctness` (no hallucination), `localization`, `impression_agreement`, `overall_clinical_utility`. Report mean + distribution.
4. **Hallucination rate** = FP / total predicted findings; **miss rate** = FN / total GT findings.
5. **Binary "clinically equivalent?"** (yes/no) → agreement rate, for a quick top-line.

### Prompt shape (describe only — implement later)
- **System:** "You are a radiology report grader. Compare a PREDICTED report to the REFERENCE. Judge only clinical content; ignore wording/formatting. Do not reward or penalize style."
- **User:** the REFERENCE and the PREDICTION, an explicit rubric, and a **strict-JSON output schema** (the counts + flags + Likert fields above). Enumerate rules: synonyms count as matches; a correctly-stated *absence* that matches the reference is neither TP nor FP; laterality/lobe mismatches are size/location errors, not misses; etc.
- **Decoding:** `--no-thinking` off is fine here (let it reason), but force a JSON block at the end and parse that. Temperature low (0–0.3) for grading stability.

### Trust the judge (calibration & bias controls)
- **Human-anchor** a small set (50–100) with radiologist/your labels; report judge-vs-human correlation (Spearman) and F1 agreement before trusting at scale.
- **Self-consistency:** sample the judge N=3–5×, average scores / majority-vote the booleans; report variance.
- **Bias controls:** reference-based (not pred-vs-pred); randomize which is "A/B" if ever doing pairwise; fixed rubric; watch length bias (judges favor longer). Consider a second judge model (e.g. a different LLM) for a subset to check judge-model bias.
- **Cost/throughput:** batch through the LMDeploy server with the resumable client pattern already in `lm_deploy/`.

---
## Cross-cutting / practical

- **Report generation failures first:** always log parse-fail / empty / truncated rates *before* scoring — a model that emits invalid JSON 20% of the time needs that surfaced, not silently dropped.
- **Consistency across heads:** the cls-token, cls-json presence, and structured-report `prominence_score` all name positive findings on the *same* test volumes — cross-check their agreement (does the free-form report mention what the classifier flags?).
- **Report-embedding metric:** you already have a CT report embedding model (`vlm/training/report_embedding`) — cosine similarity between predicted and GT report embeddings is a strong, cheap semantic score; consider it the primary automatic metric for the report task, with the LLM judge as the finding-level check.
- **Slice by cohort:** report metrics overall and broken down by source / body_part / contrast (metadata is in the masters) — e.g. does report quality drop on abdomen-inclusive scans?
- **Held-out integrity:** these test volumes are disjoint from training (verified during split creation); CT-RATE is a separate external eval set (`ct_rate_eval.parquet`) if you want an out-of-distribution report check.
- **Significance:** bootstrap CIs over volumes for the headline metrics (F1, judge-F1, embedding-sim) so model-vs-model deltas are defensible.